# 3D MRI Brain Tumor Segmentation - Quickstart

Interactive tour of the `brats_seg` package. This notebook uses **synthetic data**
so it runs end-to-end without downloading the BraTS dataset.

For real training, place the BraTS 2019 data under `data/raw/` and use the CLI
(`scripts/train.py`) as described in the README.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
from brats_seg import models, preprocessing, visualization
from brats_seg.config import Config

## 1. Build the 3D U-Net
The default configuration assumes 4 MRI modalities and 128^3 volumes.

In [ ]:
cfg = Config()
print("Resolved input shape:", cfg.resolved_input_shape())

model = models.build_brats_unet(
    input_shape=(32, 32, 32, 4),  # smaller for a quick demo
    base_filters=8,
    dropout_rate=0.0,
)
model.summary()

## 2. Preprocessing utilities
Z-score normalisation, resizing and label remapping (`4 -> 3`).

In [ ]:
rng = np.random.default_rng(0)
volume = rng.normal(50, 10, size=(40, 40, 40, 4)).astype(np.float32)
mask = np.zeros((40, 40, 40), dtype=np.uint8)
mask[15:25, 15:25, 15:25] = 4  # simulate an enhancing tumor blob

vol_p, mask_p = preprocessing.preprocess_case(
    volume, mask, target_shape=(32, 32, 32), normalize=True, remap=True
)
print("volume:", vol_p.shape, "mask:", mask_p.shape)
print("mask labels:", np.unique(mask_p))
print("per-channel mean:", vol_p.mean(axis=(0, 1, 2)).round(4))

## 3. Visualise a slice

In [ ]:
fig = visualization.show_slices(vol_p, mask_p, slice_idx=16, modality_idx=0)
fig.savefig("quickstart_slice.png", dpi=120, bbox_inches="tight")
print("saved quickstart_slice.png")

## 4. Train on real data
```bash
python scripts/train.py --data-dir data/raw --output-dir outputs --epochs 100 --loss ce
```

See the repository README for the full data layout and CLI reference.